In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("data/sql-murder-mystery.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [ ]:
# Exploring the Database Structure

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

crime_scene_report
drivers_license
person
facebook_event_checkin
interview
get_fit_now_member
get_fit_now_check_in
income
solution


In [5]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'crime_scene_report'")
for name in res:
    print(name[0])

CREATE TABLE crime_scene_report (
        date integer,
        type text,
        description text,
        city text
    )


In [6]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'drivers_license'")
for name in res:
    print(name[0])

CREATE TABLE drivers_license (
        id integer PRIMARY KEY,
        age integer,
        height integer,
        eye_color text,
        hair_color text,
        gender text,
        plate_number text,
        car_make text,
        car_model text
    )


In [7]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'person'")
for name in res:
    print(name[0])

CREATE TABLE person (
        id integer PRIMARY KEY,
        name text,
        license_id integer,
        address_number integer,
        address_street_name text,
        ssn integer,
        FOREIGN KEY (license_id) REFERENCES drivers_license(id)
    )


In [8]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'facebook_event_checkin'")
for name in res:
    print(name[0])

CREATE TABLE facebook_event_checkin (
        person_id integer,
        event_id integer,
        event_name text,
        date integer,
        FOREIGN KEY (person_id) REFERENCES person(id)
    )


In [9]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'interview'")
for name in res:
    print(name[0])

CREATE TABLE interview (
        person_id integer,
        transcript text,
        FOREIGN KEY (person_id) REFERENCES person(id)
    )


In [10]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'get_fit_now_member'")
for name in res:
    print(name[0])

CREATE TABLE get_fit_now_member (
        id text PRIMARY KEY,
        person_id integer,
        name text,
        membership_start_date integer,
        membership_status text,
        FOREIGN KEY (person_id) REFERENCES person(id)
    )


In [11]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'get_fit_now_check_in'")
for name in res:
    print(name[0])

CREATE TABLE get_fit_now_check_in (
        membership_id text,
        check_in_date integer,
        check_in_time integer,
        check_out_time integer,
        FOREIGN KEY (membership_id) REFERENCES get_fit_now_member(id)
    )


In [12]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'income'")
for name in res:
    print(name[0])

CREATE TABLE income (
        ssn integer PRIMARY KEY,
        annual_income integer
    )


In [13]:
res = crsr.execute("SELECT sql FROM sqlite_master where name = 'solution'")
for name in res:
    print(name[0])

CREATE TABLE solution (
        user integer,
        value text
    )


In [ ]:
# EJECUTAR SOLUCIÓN

In [ ]:
query = '''
INSERT INTO solution VALUES (1, 'Insert the name of the person you found here');
        
        SELECT value FROM solution;
'''

crsr.execute(query)

In [ ]:
# VISUALIZAR PISTAS

In [ ]:
# REPORTE DEL CRIMEN

In [14]:
query = '''
SELECT *
FROM crime_scene_report
WHERE city = 'SQL City'
AND type = 'murder'
AND date = 20180115;
'''

sql_query(query)

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [ ]:
# BUSCAR TESTIGOS

In [19]:
query = '''
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1;
'''

sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [24]:
query = '''
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave'
AND name LIKE '%Annabel%';
'''

sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


In [ ]:
# LEER DECLARACIONES

In [27]:
query = '''
SELECT *
FROM interview
WHERE person_id = 14887
OR person_id = 16371
'''

sql_query(query)

,person_id,transcript
0,14887,I heard a gunshot and then saw a man run out. ...
1,16371,"I saw the murder happen, and I recognized the ..."


In [ ]:
# Check data from interview transcripts

In [29]:
query = '''
SELECT *
FROM get_fit_now_member
WHERE id LIKE '%48Z%'
AND membership_status = 'gold'
'''

sql_query(query)

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


In [44]:
query = '''
SELECT 
    p.name AS Sospechoso,
    d.plate_number AS Matricula
FROM drivers_license d
JOIN person p ON p.license_id = d.id
WHERE d.plate_number LIKE '%H42W%';
'''

sql_query(query)

,Sospechoso,Matricula
0,Tushar Chandra,4H42WR
1,Jeremy Bowers,0H42W2
2,Maxine Whitely,H42W0X


In [48]:

query = '''
SELECT p.name AS ASESINO,
p.id AS ID
FROM person p
JOIN drivers_license d ON p.license_id = d.id
JOIN get_fit_now_member g ON p.id = g.person_id
WHERE d.plate_number LIKE '%H42W%'
AND g.membership_status = 'gold'
AND g.id LIKE '%48Z%';
'''

sql_query(query)

,ASESINO,ID
0,Jeremy Bowers,67318


In [50]:
# The murderer is Jeremy Bowers. But the game tells me to keep investigating...

In [49]:
query = '''
SELECT *
FROM interview
WHERE person_id = 67318
'''

sql_query(query)

,person_id,transcript
0,67318,I was hired by a woman with a lot of money. I ...


In [ ]:
# It seems that he was just the hitman... The person behind the contract is  a woman with a lot of money. 
# He doesn't know her name but: She's around 5'5" (65") or 5'7" (67"). She has red hair. She drives a Tesla Model S. 
# He knows that she attended the SQL Symphony Concert 3 times in December 2017.

In [67]:
query = '''
SELECT p.name, d.car_model, d.gender, d.hair_color, d.height, i.annual_income, f.event_name, f.date
FROM person p
JOIN drivers_license d ON p.license_id = d.id
JOIN income i ON p.ssn = i.ssn
JOIN facebook_event_checkin f ON p.id = f.person_id
WHERE d.hair_color = 'red'
AND d.car_model = 'Model S'
AND d.gender = 'female'
AND d.height BETWEEN 65 AND 67
AND f.event_name = 'SQL Symphony Concert'
'''


sql_query(query)

,name,car_model,gender,hair_color,height,annual_income,event_name,date
0,Miranda Priestly,Model S,female,red,66,310000,SQL Symphony Concert,20171206
1,Miranda Priestly,Model S,female,red,66,310000,SQL Symphony Concert,20171212
2,Miranda Priestly,Model S,female,red,66,310000,SQL Symphony Concert,20171229


In [ ]:
# The person who hired the murderer was Miranda Priestly